# StarTrek Reproducibility Analysis

This notebook allows verifying the reproducibility of training results using 5 different seeds. It generates a statistical report including the mean and the 95% confidence interval.

## 1. Imports and Configuration

## Libraries used for this test file.

- `subprocess`: to execute the training script as a sub-process.
- `re`: for regular expression parsing (to extract scores from the output).
- `math`: for mathematical operations (calculating mean and confidence intervals).
- `matplotlib`: to generate performance charts and visualization.
- `sys`: for system-specific functions (like retrieving the Python executable path).
- `os`: for file system interactions (checking if configuration files exist).
- `time`: for tracking execution time and calculating the **ETA**.

In [ ]:
import subprocess
import re
import math
import matplotlib.pyplot as plt
import sys
import os
import time

# Configuration
- `SEEDS` = the worlds where the robot will train
- `CONFIG` = path to the test configuration file
- `TRAIN_SCRIPT` = path to the training script
- `NP_EP_DEFAULT` = default number of episodes

In [ ]:
SEEDS = [89, 67, 69, 62, 99]
CONFIG = "configs/exp_fast.yaml"
TRAIN_SCRIPT = "train.py"
NP_EP_DEFAULT = 400

## Why we need to try on more than one seed.
It is essential to evaluate the AI using multiple seeds. A specific `CONFIG` might perform well on one seed by chance but poorly on others. Testing across various seeds ensures the model's robustness and consistency.

### Execution: Number of Episodes
The `get_total_episodes` function retrieves the number of episodes defined in the `.yaml` configuration file (`CONFIG`). If no value is found, it falls back to the `NP_EP_DEFAULT` constant.

### Recommendation
> It is recommended to set `NP_EP_DEFAULT` between **400** and **600** to ensure sufficient training and obtain statistically reliable results.


In [1]:
def get_total_episodes(config_path):
    try:
        path = f"../{config_path}" if not os.path.exists(config_path) else config_path
        with open(path, "r") as f:
            content = f.read()
            match = re.search(r"episodes:\s*(\d+)", content)
            if match:
                return int(match.group(1))
    except Exception:
        pass
    return NP_EP_DEFAULT

# Training Execution Logic (`run_training`)

The `run_training` function is the core of our reproducibility test. It handles the execution of the training script for a specific seed and manages the data flow.

### Key Responsibilities:
- **Isolated Execution**: It launches `train.py` as a subprocess using `subprocess.Popen`. This ensures that each training session starts with a clean state, independent of the notebook's memory.
- **Real-time Monitoring**: The function captures the script's output line by line.
- **Dynamic ETA Calculation**: For every new score collected, the function updates a local timer. It calculates the average time per episode to provide a real-time estimate of when the current seed will finish.
- **Robustness**: It monitors the process return code to ensure the training completed successfully before returning the collected scores for analysis.


In [1]:

# start the training
def run_training(seed):
    print(f"[*] Starting training with seed {seed}...")
    #take the number of episodes we want train form the config
    total_episodes = get_total_episodes(CONFIG)
    
    # Setup the subprocess.
    process = subprocess.Popen(
        [sys.executable, TRAIN_SCRIPT, CONFIG, str(seed)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd="..",
        bufsize=1
    )
    
    scores = []
    start_time = time.time()
    
    # Reading the output line by line from the subprocess
    for line in iter(process.stdout.readline, ""):
        line = line.strip()
        if not line:
            continue
            
        print(line)
        # Extracting the score from the output line using a regex.
        match = re.search(r"Score:\s*([\d\.-]+)", line)
        if match:
            try:
                scores.append(float(match.group(1)))
                # Calculating and displaying ETA for the current seed
                episodes_done = len(scores)
                elapsed = time.time() - start_time
                avg_time_per_ep = elapsed / episodes_done
                episodes_left = total_episodes - episodes_done
                eta_seconds = max(0, avg_time_per_ep * episodes_left)
                
                eta_str = time.strftime("%H:%M:%S", time.gmtime(eta_seconds))
                print(f"    -> [ETA for the end of seed {seed}: {eta_str}]")
            except Exception:
                pass

    process.stdout.close()
    process.wait()
    # Check if the process ended with an error
    if process.returncode != 0:
        print(f"[!] Training failed with code {process.returncode}")
        
    return scores

## 2. Running Training Sessions

This phase executes the training loop for each seed defined in `SEEDS`. 

### Progress Tracking & ETA
Since training 5 different models can be time-consuming, we have implemented a dual-level tracking system:
1. **Intra-seed ETA**: Displayed during each training session, it estimates the time remaining for the current seed based on the average time per episode.
2. **Global ETA**: After each completed seed, the system recalculates the estimated time remaining for all pending seeds.

> Total execution time depends heavily on `NP_EP_DEFAULT` and your hardware.

In [2]:
results = {}
start_time_total = time.time()

for i, seed in enumerate(SEEDS):
    start_time_seed = time.time()
    results[seed] = run_training(seed)
    end_time_seed = time.time()
    
    # Time calculations
    elapsed_seed = end_time_seed - start_time_seed
    elapsed_total = end_time_seed - start_time_total
    
    # Estimating remaining time
    seeds_done = i + 1
    seeds_left = len(SEEDS) - seeds_done
    avg_time_per_seed = elapsed_total / seeds_done
    eta_seconds = avg_time_per_seed * seeds_left
    
    # Display results and ETA
    eta_str = time.strftime("%H:%M:%S", time.gmtime(eta_seconds))
    print(f"[-] Seed {seed} completed in {elapsed_seed:.2f}s. ETA for remaining seeds: {eta_str}\n")


NameError: name 'time' is not defined

## 3. Statistical Analysis and Visualization

Calculation of the mean and confidence interval (**Target: 95%**)

# Chart

This chart shows the evolution of the average score over episodes, calculated from runs corresponding to different `SEEDS`.

- **Central curve**: average score at each episode.
- **Shaded area (95% CI)**: confidence interval centered on the mean to estimate variability between `SEEDS`.
- **Procedure**: all series are truncated to the same length (minimum of lengths) to compare step by step, calculating the mean and empirical standard deviation.

To estimate the final performance of each seed, we take the simple average of the **last 100 results** (or fewer if the run is shorter) to capture actual performance rather than the training phase.

> For create the graphic here we using `matplotlib`

### How to read and interpret this chart?

This chart shows the average progression of the agent over the episodes, taking into account the variability between different seeds.


- **Central curve (blue)**: this is the mean score at each episode, computed over all runs (one per seed). It gives an idea of the typical performance of the agent during training.
- **Shaded area (around the curve)**: this is the 95% confidence interval around the mean. The thinner this area, the more stable the results are from one seed to another. If it is wide, there is a lot of variability.
- **x-axis**: training episode number.
- **y-axis**: score obtained at each episode.


> If the curve rises and then stabilizes, the agent is learning properly. If the shaded area is thin, the behavior is reproducible. If it is wide, it may be useful to increase the number of seeds or episodes to get a more reliable estimate.


The final performance is read at the end of the curve: it is the mean of the last episodes, with its uncertainty.

In [ ]:
# Take all the result from the training output
all_runs = [results[s] for s in SEEDS if len(results[s]) > 0]

# check scores were collected before plotting
if not all_runs:
    print("[!] Error: No scores collected. Check if training ran correctly.")
else:
    # Trim all runs to the same length (the shortest one) to ensure proper averaging
    min_len = min(len(r) for r in all_runs)
    all_runs_trimmed = [r[:min_len] for r in all_runs]
    n = len(all_runs_trimmed)

    # Calculate mean and confidence intervals
    mean_curve = []
    for i in range(min_len):
        val_at_i = [run[i] for run in all_runs_trimmed]
        mean_curve.append(sum(val_at_i) / n)
    ci_curve = []
    # For small sample sizes, use the appropriate t-value for 95% confidence
    t_value = 2.776
    # For n=1, we can't calculate a confidence interval, so we can set it to 0 or skip it
    if n < 5:
        t_map = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776}
        t_value = t_map.get(n-1, 2.0)
    # Calculate standard deviation and confidence interval for each episode
    for i in range(min_len):
        val_at_i = [run[i] for run in all_runs_trimmed]
        m = mean_curve[i]
        variance = sum((x - m)**2 for x in val_at_i) / (n - 1) if n > 1 else 0
        std_dev = math.sqrt(variance)
        std_error = std_dev / math.sqrt(n)
        ci_curve.append(t_value * std_error)
    # Plotting the mean curve with confidence intervals
    plt.figure(figsize=(12, 6))
    episodes = list(range(min_len))
    plt.plot(episodes, mean_curve, color='#1f77b4', label='Average Score', linewidth=2)

    # Calculate upper and lower bounds for the confidence interval
    lower_bound = [m - c for m, c in zip(mean_curve, ci_curve)]
    upper_bound = [m + c for m, c in zip(mean_curve, ci_curve)]

    # Fill the area between the confidence interval bounds
    plt.fill_between(episodes, lower_bound, upper_bound, color='#1f77b4', alpha=0.2, label='95% CI')

    # Final plot adjustments
    plt.title(f'StarTrek Performance (DQN) over {n} Seeds', fontsize=14, fontweight='bold')
    plt.xlabel('Episodes', fontsize=12)
    plt.ylabel('Score', fontsize=12)
    plt.legend(frameon=True, shadow=True)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

## 4. Final Performance Report

This section summarizes the final performance for each run and provides a reproducibility estimate:

- **Window used**: we take the average of the last 100 episodes of each run (or fewer if the run is shorter) to estimate stable final performance. This focuses on the stable part of training and avoids the unstable learning phase at the beginning. It gives a better idea of the agent's real performance once it has converged.
- **Final metric**: mean ± 95% CI calculated on the final performances of the runs. The 95% confidence interval (CI) tells you how much you can trust the mean: if you repeated the experiment with new seeds, 95% of the time the true mean would fall in this interval. A small CI = stable and reliable results. A large CI = a lot of variability between runs.

Verification steps:
- If too much variance: increase the number of ``SEEDS`` or extend training to reduce standard error. If the CI is wide, you can also increase the number of episodes (longer training = more stable results. **BECAREFUL TOO MUCH THE AI LOSS MEMORY**) or change your algorithm/hyperparameters.

> Limits: This analysis only covers the tested seeds and config. It doesn't guarantee the same performance on new seeds, or in other environments. For more robustness, you can also look at the median, min/max, or plot the distribution of final scores.

> How to interpret the result? For example: if the final report says `Final Performance: 200 + 15`, it means you can expect the agent to reach a score between ~185 and ~215 on a new run, 95% of the time.

In [ ]:
if not all_runs:
    print("[!] No final report possible.")
else:
    final_performances = []
    for run in all_runs:
        # Taking the last 100 episodes
        window = min(100, len(run))
        last_window = run[-window:]
        if last_window:
            final_performances.append(sum(last_window) / len(last_window))
    # Calculate mean and confidence interval for the final performance
    if final_performances:
        n_final = len(final_performances)
        final_mean = sum(final_performances) / n_final
        # Calculate variance and standard error for the final performance
        final_variance = sum((x - final_mean)**2 for x in final_performances) / (n_final - 1) if n_final > 1 else 0
        final_std_error = math.sqrt(final_variance) / math.sqrt(n_final)
        
        # For small sample sizes, use the appropriate t-value for 95% confidence
        t_map = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776}
        t_val = t_map.get(n_final-1, 2.0)
        final_ci = t_val * final_std_error

        # Print the final report
        print("="*40)
        print(" REPRODUCIBILITY REPORT ")
        print("="*40)
        print(f"Configuration: {CONFIG}")
        print(f"Successful runs: {n_final}/{len(SEEDS)}")
        print(f"Final Performance: {final_mean:.2f} ± {final_ci:.2f}")
        print("="*40)
    else:
        print("[!] Not enough data for the final report.")